# Aosta Valley Temperature Prediction — Kriging + Transformer / TCN

This notebook covers:
1. Ordinary and Universal Kriging with numerical-stability error handling
2. Transformer and TCN models trained and evaluated on separate splits

**Bug fixes applied in this notebook:**
- **Issue 5** (lines 38-56): `OK.execute` / `UK.execute` wrapped in
  `try/except (LinAlgError, ValueError, RuntimeError)` via the
  `run_ordinary_kriging` / `run_universal_kriging` helpers.
- **Issue 4** (lines 81-96): Transformer and TCN are fitted **only** on
  the training partition of `(X_reshaped, y_reshaped)` — not on the full
  dataset that is also used for evaluation.
- **Issue 2** (lines 83-117): MSE, MAE, and R² are computed exclusively
  on the held-out test partition, never on the training data.

## 0. Imports and Setup

In [ ]:
import sys
sys.path.append('..')

import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from numpy.linalg import LinAlgError

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from src.data.loader import load_config
from src.data.preprocessor import build_pipeline
from src.models.kriging import run_ordinary_kriging, run_universal_kriging
from src.models.ml_models import prepare_features
from src.utils.metrics import evaluate_model

config = load_config('../config.yaml')
print('Config loaded.')

## 1. Load and Preprocess Data

In [ ]:
station_gdf = build_pipeline(config, base_path='..')
print(f'Dataset shape: {station_gdf.shape}')

## 2. Spatial Interpolation — Ordinary and Universal Kriging

### Issue 5 Fix (lines 38–56)
`OK.execute` and `UK.execute` perform Cholesky / LU decompositions on the
semi-variogram covariance matrix.  When that matrix is near-singular (e.g.,
stations too close together or a poorly-fitted variogram), NumPy raises
`LinAlgError`.  The `run_ordinary_kriging` / `run_universal_kriging` helpers
catch `(LinAlgError, ValueError, RuntimeError)` and return `(None, None)`,
keeping the notebook runnable regardless of data quality.

In [ ]:
# Aggregate: one mean temperature per unique station location
station_snapshot = (
    station_gdf
    .groupby(['station_id', 'longitude', 'latitude'], as_index=False)
    ['temperature'].mean()
    .dropna()
)

x_coords = station_snapshot['longitude'].values
y_coords = station_snapshot['latitude'].values
z_values = station_snapshot['temperature'].values

grid_x = np.linspace(x_coords.min(), x_coords.max(), 50)
grid_y = np.linspace(y_coords.min(), y_coords.max(), 50)

print(f'Stations: {len(x_coords)}  |  Grid: {len(grid_x)}x{len(grid_y)}')

In [ ]:
# ---------------------------------------------------------------------------
# Issue 5 fix (lines 38-56):
# Both helpers wrap OK.execute / UK.execute in
#   try/except (LinAlgError, ValueError, RuntimeError)
# and emit a RuntimeWarning instead of raising, returning (None, None)
# so downstream cells can check and continue safely.
# ---------------------------------------------------------------------------

ok_pred, ok_var = run_ordinary_kriging(
    x_coords, y_coords, z_values,
    grid_x, grid_y,
    variogram_model='linear',
)
print('OK result:', 'success' if ok_pred is not None else 'failed (see warning)')

uk_pred, uk_var = run_universal_kriging(
    x_coords, y_coords, z_values,
    grid_x, grid_y,
    variogram_model='linear',
)
print('UK result:', 'success' if uk_pred is not None else 'failed (see warning)')

## 3. Prepare Features for Deep Learning

### Issues 2 & 4 Fix: Train / Test Split
The original notebook computed `X_reshaped` / `y_reshaped` from the full
dataset, then fitted *and* evaluated the Transformer and TCN on the
exact same array.  Every metric was therefore a trivially optimistic
in-sample score rather than a generalisation estimate.

Fix: split into `(X_train_reshaped, X_test_reshaped)` before `model.fit`,
and compute MSE / MAE / R² only on `X_test_reshaped`.

In [ ]:
# -------------------------------------------------------------------------
# Build numeric feature matrix and target vector
# -------------------------------------------------------------------------
target_col = config['features']['target']
drop_cols  = config['features']['drop_cols']

y_series = station_gdf[target_col].dropna()
X_df     = station_gdf.drop(columns=drop_cols).loc[y_series.index]
X_df     = X_df.select_dtypes(include='number')
X_df     = X_df.apply(lambda col: col.fillna(col.median()))

X_array = X_df.values.astype(np.float32)
y_array = y_series.values.astype(np.float32)

# -------------------------------------------------------------------------
# Issue 4 & 2 fix (lines 81-117): split BEFORE fitting, evaluate ONLY
# on the held-out test set.
# -------------------------------------------------------------------------
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_array, y_array,
    test_size=0.2,
    random_state=42,
)

# Scale features — fit on training data ONLY to avoid data leakage
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_raw)
X_test_scaled  = scaler.transform(X_test_raw)

n_features = X_train_scaled.shape[1]

print(f'Train: {X_train_scaled.shape}  |  Test: {X_test_scaled.shape}')

## 4. Transformer Encoder

### Issue 4 Fix — no training on the full dataset

In [ ]:
try:
    import tensorflow as tf
    _TF_AVAILABLE = True
except ImportError:
    _TF_AVAILABLE = False
    print('TensorFlow not installed — skipping Transformer and TCN cells.')

In [ ]:
if _TF_AVAILABLE:
    tf_cfg = config.get('models', {}).get('transformer', {})
    d_model    = tf_cfg.get('d_model', 32)
    num_heads  = tf_cfg.get('num_heads', 2)
    num_layers = tf_cfg.get('num_layers', 2)
    dropout_r  = tf_cfg.get('dropout', 0.1)
    epochs_tf  = tf_cfg.get('epochs', 30)
    batch_tf   = tf_cfg.get('batch_size', 64)

    # -----------------------------------------------------------------------
    # Issue 4 fix (lines 81-96): reshape uses the TRAINING split only.
    # -----------------------------------------------------------------------
    # Reshape: (samples, seq_len=n_features, channels=1)
    X_train_reshaped = X_train_scaled[:, :, np.newaxis]  # (N_train, F, 1)
    X_test_reshaped  = X_test_scaled[:, :, np.newaxis]   # (N_test,  F, 1)

    # Model definition
    inputs = tf.keras.Input(shape=(n_features, 1))
    x = tf.keras.layers.Dense(d_model)(inputs)

    for _ in range(num_layers):
        attn = tf.keras.layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=max(d_model // num_heads, 1),
        )(x, x)
        attn = tf.keras.layers.Dropout(dropout_r)(attn)
        x    = tf.keras.layers.LayerNormalization(epsilon=1e-6)(x + attn)
        ff   = tf.keras.layers.Dense(d_model * 2, activation='relu')(x)
        ff   = tf.keras.layers.Dense(d_model)(ff)
        ff   = tf.keras.layers.Dropout(dropout_r)(ff)
        x    = tf.keras.layers.LayerNormalization(epsilon=1e-6)(x + ff)

    x       = tf.keras.layers.GlobalAveragePooling1D()(x)
    outputs = tf.keras.layers.Dense(1)(x)

    transformer_model = tf.keras.Model(inputs=inputs, outputs=outputs)
    transformer_model.compile(optimizer='adam', loss='mse')

    early_stop = tf.keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=5, restore_best_weights=True
    )

    # -----------------------------------------------------------------------
    # Issue 4 fix: fit ONLY on the training split.
    # -----------------------------------------------------------------------
    transformer_model.fit(
        X_train_reshaped, y_train,
        epochs=epochs_tf,
        batch_size=batch_tf,
        validation_split=0.1,
        callbacks=[early_stop],
        verbose=0,
    )

    # -----------------------------------------------------------------------
    # Issue 2 fix (lines 83-117): evaluate on the held-out TEST set.
    # Previously evaluation used X_reshaped (the full dataset = training
    # data), yielding inflated / meaningless metrics.
    # -----------------------------------------------------------------------
    y_pred_tf = transformer_model.predict(X_test_reshaped, verbose=0).flatten()

    tf_result = evaluate_model(y_test, y_pred_tf, model_name='Transformer')
    print('Transformer result:', tf_result)

## 5. Temporal Convolutional Network (TCN)

### Issue 4 Fix — no training on the full dataset

In [ ]:
if _TF_AVAILABLE:
    tcn_cfg    = config.get('models', {}).get('tcn', {})
    filters    = tcn_cfg.get('filters', 64)
    kernel_sz  = tcn_cfg.get('kernel_size', 3)
    num_blocks = tcn_cfg.get('num_blocks', 3)
    dropout_r  = tcn_cfg.get('dropout', 0.1)
    epochs_tcn = tcn_cfg.get('epochs', 30)
    batch_tcn  = tcn_cfg.get('batch_size', 64)

    # -----------------------------------------------------------------------
    # Issue 4 fix (lines 81-96): reshape uses the TRAINING split only.
    # TCN expects (samples, timesteps=1, channels=n_features)
    # -----------------------------------------------------------------------
    X_train_tcn = X_train_scaled[:, np.newaxis, :]  # (N_train, 1, F)
    X_test_tcn  = X_test_scaled[:, np.newaxis, :]   # (N_test,  1, F)

    # Dilated causal Conv1D blocks
    inputs = tf.keras.Input(shape=(1, n_features))
    x = inputs
    for i in range(num_blocks):
        dilation = 2 ** i
        residual = x
        x = tf.keras.layers.Conv1D(
            filters, kernel_sz, padding='causal',
            dilation_rate=dilation, activation='relu'
        )(x)
        x = tf.keras.layers.SpatialDropout1D(dropout_r)(x)
        x = tf.keras.layers.Conv1D(
            filters, kernel_sz, padding='causal',
            dilation_rate=dilation, activation='relu'
        )(x)
        x = tf.keras.layers.SpatialDropout1D(dropout_r)(x)
        if residual.shape[-1] != filters:
            residual = tf.keras.layers.Conv1D(filters, 1)(residual)
        x = tf.keras.layers.Add()([x, residual])
        x = tf.keras.layers.LayerNormalization()(x)

    x       = tf.keras.layers.GlobalAveragePooling1D()(x)
    outputs = tf.keras.layers.Dense(1)(x)

    tcn_model = tf.keras.Model(inputs=inputs, outputs=outputs)
    tcn_model.compile(optimizer='adam', loss='mse')

    early_stop_tcn = tf.keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=5, restore_best_weights=True
    )

    # -----------------------------------------------------------------------
    # Issue 4 fix: fit ONLY on the training split.
    # -----------------------------------------------------------------------
    tcn_model.fit(
        X_train_tcn, y_train,
        epochs=epochs_tcn,
        batch_size=batch_tcn,
        validation_split=0.1,
        callbacks=[early_stop_tcn],
        verbose=0,
    )

    # -----------------------------------------------------------------------
    # Issue 2 fix (lines 83-117): evaluate on the held-out TEST set.
    # -----------------------------------------------------------------------
    y_pred_tcn = tcn_model.predict(X_test_tcn, verbose=0).flatten()

    tcn_result = evaluate_model(y_test, y_pred_tcn, model_name='TCN')
    print('TCN result:', tcn_result)

## 6. Results Summary

In [ ]:
if _TF_AVAILABLE:
    results = [tf_result, tcn_result]
    summary_df = (
        pd.DataFrame(results)
        .set_index('model')
        .sort_values('r2', ascending=False)
    )
    print(summary_df.to_string())
    summary_df.style.format(
        {'mse': '{:.4f}', 'mae': '{:.4f}', 'r2': '{:.4f}'}
    ).background_gradient(subset=['r2'], cmap='RdYlGn')